In [3]:
import os 
from dotenv import load_dotenv 
from langchain.agents import create_agent 
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from crawl4ai import AsyncWebCrawler 
import asyncio 
from langchain_openrouter import ChatOpenRouter
from pydantic import BaseModel, Field
import json

load_dotenv()

## for now its text
user_query = input ("what crisis are you facing?")
model = ChatOpenRouter(model="nvidia/nemotron-3-nano-30b-a3b:free" , temperature=0.8)

class JsonClass(BaseModel):
    Topic : str = Field(description= "this will be the topic for which the user is facing crisis")
    Locationquery : str = Field(description = "you have to respond with the exact query i can put on google maps in order to get places for help with respect to that query")
    confidence_threshold : float = Field(description= "how confident do you think your classification is out of 0-1")

schema = json.dumps(JsonClass.model_json_schema() , indent = 2)

topics_helpline = [
    "Abuse & domestic violence",
    "Anxiety",
    "Bullying",
    "Dementia & Alzheimers",
    "Depression",
    "Eating & body image",
    "Family",
    "Gambling",
    "Gender & sexual identity",
    "Grief & loss",
    "Loneliness",
    "Parenting",
    "Physical illness",
    "Pregnancy & abortion",
    "Relationships",
    "School & work",
    "Self-harm",
    "Sexual abuse",
    "Stress",
    "Substance use",
    "Suicide",
    "Supporting others",
    "Trauma & PTSD"
]


template = ChatPromptTemplate.from_messages([
    (
        "system",
        """you are a user query classifier assistant that takes in user query and responds in this exact json schema {schema}. the available topics are: {topics}. this will help us get to know what type of crisis he/she is in and also what are the locations he can reach out for help

        Rules:
        - always stick to the json schema provided
        - always rate your output out of 0-1 . if confidence threshold is below 0.6 , respond with none
        - respond with valid JSON only, no markdown or extra text"""
    ),
    (
        "human",
        "{user_input}"
    )
])

chain = template | model
response = chain.invoke({
    "schema": schema,
    "topics": topics_helpline,
    "user_input": user_query
})
llm_output = response.content.strip()
if llm_output.startswith("```"):
    llm_output = llm_output.strip("`").removeprefix("json").strip()
parsed = json.loads(llm_output)
print(f"Topic: {parsed['Topic']}")
print(f"Location query: {parsed['Locationquery']}")
print(f"Confidence: {parsed['confidence_threshold']}")


async def get_places_forhelp(llm_ouput : str ) -> list[dict] : 
    url = f"https://findahelpline.com/countries/in/topics/{llm_ouput}"
    async with AsyncWebCrawler() as crawler :
        result = await crawler.arun(url=url)
        print(result.markdown)

asyncio.run(get_places_forhelp())






Topic: Loneliness
Location query: loneliness support groups near me
Confidence: 0.93


ValueError: Function must have a docstring if description not provided.

In [ ]:
helplines = {
    # Emergency
    "112": "Integrated Emergency Response (Police, Fire, Health, Rescue)",
    "102": "National Ambulance Service",
    "100": "Police Helpline",
    "101": "Fire Helpline",
    "1930": "Cyber Crime Helpline",
    "1906": "LPG Leak Helpline",
    "1070": "Natural Calamities Relief Commissioner",
    "011-24363260": "Earthquake/Flood/Disaster/NDRF",
    "1915": "National Consumer Helpline",
    "1073": "Road Accident Helpline",
    "1933": "National Narcotics Helpline (MANAS)",

    # Women
    "7827170170": "National Commission for Women Helpline",
    "10920": "Shakti Shalini Helpline",
    "011-24619821": "Sakhi Women's Helpline",
    "181": "Women Helpline / Domestic Violence Helpline",
    "1800-2000-737": "Rape Crisis Helpline",
    "14678": "SUBHADRA Yojana Helpline",

    # Children
    "1098": "Child Helpline",
    "14433": "National Human Rights Commission Toll-Free Helpline",
    "1091": "Anti-Obscene Calls Cell",

    # Senior Citizens / Disabilities
    "011-20893999": "Senior Citizens Helpline",
    "14567": "Senior Citizens Helpline",
    "14456": "Persons with Disabilities Helpline",

    # Enquiries
    "139": "Railway Security and Medical Assistance",
    "3800": "State Bank of India Helpline",
    "0120-6619540": "National Scholarship Portal Technical Helpdesk",
    "1800-258-1800": "Passport Seva Helpline",
    "040-66720567": "Passport Seva (Jammu & Kashmir)",
    "040-66720581": "Passport Seva (North-East States)",

    # Other
    "011-23978046": "COVID-19 Central Helpline",
    "1075": "COVID-19 Toll-Free Helpline",
    "1800-11-1363": "Tourist Helpline",
    "1363": "Tourist Helpline (Short Code)",
    "1947": "UIDAI (Aadhaar) Toll-Free Helpline",
    "1800110396": "PM Daksh Helpline",
    "1800112100": "FSSAI Helpdesk"
}


<built-in method keys of dict object at 0x1151300c0>


In [9]:
import serpapi
import os 
from dotenv import load_dotenv

load_dotenv()

serp_apikey = os.getenv("SERP_API_KEY")
if not serp_apikey :
    print('no api key found')

client = serpapi.Client(api_key=serp_apikey)
results = client.search({
  "engine": "google_maps",
  "q": "Coffee",
  "ll": "@40.7455096,-74.0083012,14z"
})
local_results = results["local_results"]
print(local_results)

[{'position': 1, 'title': 'Le Cafe Coffee', 'place_id': 'ChIJ-Y-kFABZwokRIpR1-ZpuRjQ', 'data_id': '0x89c2590014a48ff9:0x34466e9af9759422', 'data_cid': '3766819750231249954', 'reviews_link': 'https://serpapi.com/search.json?data_id=0x89c2590014a48ff9%3A0x34466e9af9759422&engine=google_maps_reviews&hl=en', 'photos_link': 'https://serpapi.com/search.json?data_id=0x89c2590014a48ff9%3A0x34466e9af9759422&engine=google_maps_photos&hl=en', 'gps_coordinates': {'latitude': 40.765291, 'longitude': -73.9767121}, 'place_id_search': 'https://serpapi.com/search.json?engine=google_maps&google_domain=google.com&hl=en&place_id=ChIJ-Y-kFABZwokRIpR1-ZpuRjQ', 'provider_id': '/g/11ld95sfh0', 'rating': 4.6, 'reviews': 259, 'price': '$1–10', 'extracted_price': 1, 'type': 'Coffee shop', 'types': ['Coffee shop'], 'type_id': 'coffee_shop', 'type_ids': ['coffee_shop'], 'address': '1427 6th Ave, New York, NY 10019', 'country': 'United States', 'open_state': 'Opens soon · 7\u202fAM', 'hours': 'Opens soon · 7\u202fA

In [10]:
import os 
from dotenv import load_dotenv 
from langchain.agents import create_agent 
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from crawl4ai import AsyncWebCrawler 
import asyncio 
from langchain_openrouter import ChatOpenRouter
from pydantic import BaseModel, Field
import json
import serpapi

load_dotenv()

## for now its text
user_query = input ("what crisis are you facing?")
model = ChatOpenRouter(model="nvidia/nemotron-3-nano-30b-a3b:free" , temperature=0.8)

class JsonClass(BaseModel):
    Topic : str = Field(description= "this will be the topic for which the user is facing crisis")
    location : str = Field(description = "the city or location mentioned by the user in their query. if no location mentioned, output empty string")
    Locationquery : str = Field(description = "the exact query to enter in google maps to find nearby help places, combining the topic and location. make sure to use + symbol to seperate words as that is the format accepted by gmaps. ")
    need_for_nationalhelpline : bool = Field(description="if the user query needs a national helpline assistance , true , else false ")
    national_assistance_which : str = Field(description = "which national assistance helpline is needed for the user query . only if needed. else output none ")
    national_assistance_helpline_number : int = Field(description="the phone number of the helpline (national one) that solves the problem of user query/crisis/ only if it exists . else none")
    confidence_threshold : float = Field(description= "how confident do you think your classification is out of 0-1")

schema = json.dumps(JsonClass.model_json_schema() , indent = 2)

topics_helpline = [
    "Abuse & domestic violence",
    "Anxiety",
    "Bullying",
    "Dementia & Alzheimers",
    "Depression",
    "Eating & body image",
    "Family",
    "Gambling",
    "Gender & sexual identity",
    "Grief & loss",
    "Loneliness",
    "Parenting",
    "Physical illness",
    "Pregnancy & abortion",
    "Relationships",
    "School & work",
    "Self-harm",
    "Sexual abuse",
    "Stress",
    "Substance use",
    "Suicide",
    "Supporting others",
    "Trauma & PTSD"
]
helplines = {
    # Emergency
    "112": "Integrated Emergency Response (Police, Fire, Health, Rescue)",
    "102": "National Ambulance Service",
    "100": "Police Helpline",
    "101": "Fire Helpline",
    "1930": "Cyber Crime Helpline",
    "1906": "LPG Leak Helpline",
    "1070": "Natural Calamities Relief Commissioner",
    "011-24363260": "Earthquake/Flood/Disaster/NDRF",
    "1915": "National Consumer Helpline",
    "1073": "Road Accident Helpline",
    "1933": "National Narcotics Helpline (MANAS)",

    # Women
    "7827170170": "National Commission for Women Helpline",
    "10920": "Shakti Shalini Helpline",
    "011-24619821": "Sakhi Women's Helpline",
    "181": "Women Helpline / Domestic Violence Helpline",
    "1800-2000-737": "Rape Crisis Helpline",
    "14678": "SUBHADRA Yojana Helpline",

    # Children
    "1098": "Child Helpline",
    "14433": "National Human Rights Commission Toll-Free Helpline",
    "1091": "Anti-Obscene Calls Cell",

    # Senior Citizens / Disabilities
    "011-20893999": "Senior Citizens Helpline",
    "14567": "Senior Citizens Helpline",
    "14456": "Persons with Disabilities Helpline",

    # Enquiries
    "139": "Railway Security and Medical Assistance",
    "3800": "State Bank of India Helpline",
    "0120-6619540": "National Scholarship Portal Technical Helpdesk",
    "1800-258-1800": "Passport Seva Helpline",
    "040-66720567": "Passport Seva (Jammu & Kashmir)",
    "040-66720581": "Passport Seva (North-East States)",

    # Other
    "011-23978046": "COVID-19 Central Helpline",
    "1075": "COVID-19 Toll-Free Helpline",
    "1800-11-1363": "Tourist Helpline",
    "1363": "Tourist Helpline (Short Code)",
    "1947": "UIDAI (Aadhaar) Toll-Free Helpline",
    "1800110396": "PM Daksh Helpline",
    "1800112100": "FSSAI Helpdesk"
}

template = ChatPromptTemplate.from_messages([
    (
        "system",
        """you are a user query classifier assistant that takes in user query and responds in this exact json schema {schema}. the available topics are: {topics}. this will help us get to know what type of crisis he/she is in and also what are the locations he can reach out for help

        Rules:
        - always stick to the json schema provided
        - always rate your output out of 0-1 . if confidence threshold is below 0.6 , respond with none
        - respond with valid JSON only, no markdown or extra text
        - if user query needs national helpline assistance only then output for all the national params in the schema . else output None
        - use the helplines dict to fill national_assistance_which and national_assistance_helpline_number: {helplines}"""
    ),
    (
        "human",
        "{user_input}"
    )
])

chain = template | model
response = chain.invoke({
    "schema": schema,
    "topics": topics_helpline,
    "helplines": json.dumps(helplines, indent=2),
    "user_input": user_query
})
llm_output = response.content.strip()
if llm_output.startswith("```"):
    llm_output = llm_output.strip("`").removeprefix("json").strip()
parsed = json.loads(llm_output)
topic = parsed['Topic']
location = parsed['location']
location_query = parsed['Locationquery']
need_national_helpline = parsed['need_for_nationalhelpline']
national_helpline_type = parsed['national_assistance_which']
national_helpline_number = parsed['national_assistance_helpline_number']
confidence = parsed['confidence_threshold']

print(f"Topic: {topic}")
print(f"Location: {location}")
print(f"Location query (Google Maps): {location_query}")
print(f"Need national helpline: {need_national_helpline}")
print(f"National assistance type: {national_helpline_type}")
print(f"National helpline number: {national_helpline_number}")
print(f"Confidence: {confidence}")

Topic: Stress
Location: 
Location query (Google Maps): Stress
Need national helpline: True
National assistance type: National Consumer Helpline
National helpline number: 1915
Confidence: 0.96


In [1]:
import os 
from dotenv import load_dotenv 
from langchain.agents import create_agent 
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from crawl4ai import AsyncWebCrawler 
import asyncio 
from langchain_openrouter import ChatOpenRouter
from pydantic import BaseModel, Field
import json
import serpapi

load_dotenv()

## for now its text
user_query = input ("what crisis are you facing?")
model = ChatOpenRouter(model="nvidia/nemotron-3-nano-30b-a3b:free" , temperature=0.8)

class JsonClass(BaseModel):
    Topic : str = Field(description= "this will be the topic for which the user is facing crisis")
    location : str = Field(description = "the city or location mentioned by the user in their query. output only the first two letters of the state of the user. if no location mentioned, output empty string")
    Locationquery : str = Field(description = "the exact query to enter in google maps to find nearby help places, combining the topic and location. make sure to use + symbol to seperate words as that is the format accepted by gmaps. ")
    need_for_nationalhelpline : bool = Field(description="if the user query needs a national helpline assistance , true , else false ")
    national_assistance_which : str = Field(description = "which national assistance helpline is needed for the user query . only if needed. else output none ")
    national_assistance_helpline_number : int = Field(description="the phone number of the helpline (national one) that solves the problem of user query/crisis/ only if it exists . else none")
    confidence_threshold : float = Field(description= "how confident do you think your classification is out of 0-1")

schema = json.dumps(JsonClass.model_json_schema() , indent = 2)

topics_helpline = [
    "Abuse & domestic violence",
    "Anxiety",
    "Bullying",
    "Dementia & Alzheimers",
    "Depression",
    "Eating & body image",
    "Family",
    "Gambling",
    "Gender & sexual identity",
    "Grief & loss",
    "Loneliness",
    "Parenting",
    "Physical illness",
    "Pregnancy & abortion",
    "Relationships",
    "School & work",
    "Self-harm",
    "Sexual abuse",
    "Stress",
    "Substance use",
    "Suicide",
    "Supporting others",
    "Trauma & PTSD"
]
helplines = {
    # Emergency
    "112": "Integrated Emergency Response (Police, Fire, Health, Rescue)",
    "102": "National Ambulance Service",
    "100": "Police Helpline",
    "101": "Fire Helpline",
    "1930": "Cyber Crime Helpline",
    "1906": "LPG Leak Helpline",
    "1070": "Natural Calamities Relief Commissioner",
    "011-24363260": "Earthquake/Flood/Disaster/NDRF",
    "1915": "National Consumer Helpline",
    "1073": "Road Accident Helpline",
    "1933": "National Narcotics Helpline (MANAS)",

    # Women
    "7827170170": "National Commission for Women Helpline",
    "10920": "Shakti Shalini Helpline",
    "011-24619821": "Sakhi Women's Helpline",
    "181": "Women Helpline / Domestic Violence Helpline",
    "1800-2000-737": "Rape Crisis Helpline",
    "14678": "SUBHADRA Yojana Helpline",

    # Children
    "1098": "Child Helpline",
    "14433": "National Human Rights Commission Toll-Free Helpline",
    "1091": "Anti-Obscene Calls Cell",

    # Senior Citizens / Disabilities
    "011-20893999": "Senior Citizens Helpline",
    "14567": "Senior Citizens Helpline",
    "14456": "Persons with Disabilities Helpline",

    # Enquiries
    "139": "Railway Security and Medical Assistance",
    "3800": "State Bank of India Helpline",
    "0120-6619540": "National Scholarship Portal Technical Helpdesk",
    "1800-258-1800": "Passport Seva Helpline",
    "040-66720567": "Passport Seva (Jammu & Kashmir)",
    "040-66720581": "Passport Seva (North-East States)",

    # Other
    "011-23978046": "COVID-19 Central Helpline",
    "1075": "COVID-19 Toll-Free Helpline",
    "1800-11-1363": "Tourist Helpline",
    "1363": "Tourist Helpline (Short Code)",
    "1947": "UIDAI (Aadhaar) Toll-Free Helpline",
    "1800110396": "PM Daksh Helpline",
    "1800112100": "FSSAI Helpdesk"
}

template = ChatPromptTemplate.from_messages([
    (
        "system",
        """you are a user query classifier assistant that takes in user query and responds in this exact json schema {schema}. the available topics are: {topics}. this will help us get to know what type of crisis he/she is in and also what are the locations he can reach out for help

        Rules:
        - always stick to the json schema provided
        - always rate your output out of 0-1 . if confidence threshold is below 0.6 , respond with none
        - respond with valid JSON only, no markdown or extra text
        - if user query needs national helpline assistance only then output for all the national params in the schema . else output None
        - use the helplines dict to fill national_assistance_which and national_assistance_helpline_number: {helplines}"""
    ),
    (
        "human",
        "{user_input}"
    )
])

chain = template | model
response = chain.invoke({
    "schema": schema,
    "topics": topics_helpline,
    "helplines": json.dumps(helplines, indent=2),
    "user_input": user_query
})
llm_output = response.content.strip()
if llm_output.startswith("```"):
    llm_output = llm_output.strip("`").removeprefix("json").strip()
parsed = json.loads(llm_output)
topic = parsed['Topic']
location = parsed['location']
location_query = parsed['Locationquery']
need_national_helpline = parsed['need_for_nationalhelpline']
national_helpline_type = parsed['national_assistance_which']
national_helpline_number = parsed['national_assistance_helpline_number']
confidence = parsed['confidence_threshold']

print(f"Topic: {topic}")
print(f"Location: {location}")
print(f"Location query (Google Maps): {location_query}")
print(f"Need national helpline: {need_national_helpline}")
print(f"National assistance type: {national_helpline_type}")
print(f"National helpline number: {national_helpline_number}")
print(f"Confidence: {confidence}")

async def get_help_forquery(topic: str, location_query: str):
    """Crawls findahelpline.com for crisis resources and searches Google Maps via SerpAPI for nearby help places."""
    helpline_find_url = f"https://findahelpline.com/countries/in/{location}/topics/{topic}"
    async with AsyncWebCrawler() as crawler : 
        result = await crawler.arun(url = helpline_find_url)
        print(result.markdown)

    serp_key = os.getenv("SERP_API_KEY")
    client = serpapi.Client(api_key=serp_key)
    results = client.search({
        "engine": "google_maps",
        "q": location_query,
        "type": "search"
    })
    print("\n=== Top Places Nearby ===")
    for i, place in enumerate(results.get("local_results", [])[:10], 1):
        print(f"{i}. {place.get('title', 'N/A')}")
        print(f"   Phone: {place.get('phone', 'N/A')}")
        print(f"   Website: {place.get('website', 'N/A')}")
        print()

asyncio.run(get_help_forquery(topic, location_query))





Topic: Family
Location: 
Location query (Google Maps): Family
Need national helpline: True
National assistance type: National Consumer Helpline
National helpline number: 1915
Confidence: 0.78


RuntimeError: asyncio.run() cannot be called from a running event loop